# Predictive Alerting for Cloud Metrics

This notebook covers the full ML pipeline:

1. Problem formulation and dataset
2. Feature engineering rationale
3. Model selection and training
4. Evaluation with results analysis
5. Failure case analysis and limitations

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 4)

## 1. Problem Formulation

### Sliding-window binary classification

For each timestamp *t*:
- **Input**: feature vector derived from observations at *[t−W+1, …, t]*
- **Target**: `1` if any incident occurs in *(t, t+H]*, else `0`

We use **W = 60 minutes** and **H = 15 minutes**.

This framing converts the time-series problem into standard tabular classification, which lets us use XGBoost without needing sequence models. The 15-minute horizon is long enough to act on, short enough that predictions remain informative.

### Dataset

Synthetic CloudWatch-like metrics (CPU, memory, latency, error rate, network I/O) with injected incidents. Incidents are correlated multi-metric spikes with realistic ramp-up profiles and random severity. On real data, labels would come from existing CloudWatch alarms entering ALARM state.

In [ ]:
from src.data.generator import generate_dataset

df = generate_dataset(n_days=90, freq_minutes=1, incident_rate=0.02, seed=42)

n_incidents = int((df['incident'].astype(int).diff() == 1).sum())
mean_duration = df['incident'].sum() / n_incidents if n_incidents else 0

print(f'Rows: {len(df):,}  |  Period: {df.index.min().date()} – {df.index.max().date()}')
print(f'Incidents: {n_incidents}  |  Mean duration: {mean_duration:.1f} min  |  Incident rate: {df["incident"].mean():.2%}')

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 9))
metric_cols = ['cpu_utilization', 'memory_usage', 'request_latency', 'error_rate', 'network_in', 'network_out']
sample = df.iloc[:2880]

for ax, col in zip(axes.flat, metric_cols):
    ax.plot(sample.index, sample[col], linewidth=0.7, alpha=0.85)
    ax.fill_between(sample.index, sample[col].min(), sample[col].max(),
                    where=sample['incident'], alpha=0.25, color='red')
    ax.set_title(col.replace('_', ' ').title())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %Hh'))
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right', fontsize=8)

fig.legend([plt.Rectangle((0,0),1,1,fc='red',alpha=0.25)], ['Incident'], loc='upper right')
fig.suptitle('First 2 days of synthetic data — incident intervals in red', fontsize=13)
plt.tight_layout()
plt.show()

**Observations:** All metrics show diurnal patterns. During incidents, CPU, latency, and error rate spike simultaneously — cross-metric correlation is a useful signal. Distributions are right-skewed (especially `request_latency`), which motivates percentile-based features rather than raw means.

## 2. Feature Engineering

Tree-based models require fixed-length feature vectors, not raw sequences. We summarize the W-step history using:

| Group | Captures | Why |
|---|---|---|
| Rolling mean/std/min/max over [5, 15, 30, 60 min] | Trend and volatility at multiple scales | Short windows catch spikes; long windows catch slow drifts |
| Lag values [1, 5, 15, 30 min] | Recent level and momentum | Direct window observations |
| Rate-of-change (diff, pct_change) | How fast metrics are moving | Velocity is more predictive than absolute level |
| Percentile features (q25, q75, q95, IQR) | Tail behavior | IQR is outlier-resistant; q95 captures heavy-tail spikes |
| Cyclical time (hour, day-of-week) | Seasonality | Prevents treating hour 23 and hour 0 as distant |

In [ ]:
from src.data.preprocessing import create_features, create_targets, split_temporal

features = create_features(df, window_size=60, windows=[5, 15, 30, 60], lags=[1, 5, 15, 30])
targets = create_targets(df.loc[features.index], lookahead_minutes=15)
features = features.loc[targets.index]
targets = targets.loc[features.index]

print(f'Feature matrix: {features.shape[0]:,} × {features.shape[1]}')
print(f'Target positive rate: {targets.mean():.2%}  (class imbalance ~1:{int(1/targets.mean())})')

In [ ]:
selected = ['cpu_utilization', 'error_rate', 'request_latency',
            'cpu_utilization_mean_15m', 'error_rate_roc_1m',
            'request_latency_q95_60m', 'cpu_utilization_std_15m']
selected = [c for c in selected if c in features.columns]
combined = features[selected].copy()
combined['target'] = targets.values

plt.figure(figsize=(9, 7))
sns.heatmap(combined.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature–target correlations (subset)', fontsize=12)
plt.tight_layout()
plt.show()

**`error_rate_roc_1m`** and **`request_latency_q95_60m`** show higher correlation with the target than raw values — confirming that velocity and tail behavior are more predictive than current absolute values.

## 3. Model Selection and Training

### Why not LSTM / Transformer?
Sequence models are a reasonable alternative but require more data and compute, are harder to retrain frequently, and offer no interpretability benefit. For tabular time-series features, XGBoost typically matches LSTM performance with much simpler training.

### Why an ensemble of IsolationForest + XGBoost?

**IsolationForest** is trained only on normal data. It detects deviations from the current baseline without requiring labels — useful when labels are sparse or when novel failure modes appear.
Weakness: high false positive rate on routine traffic spikes.

**XGBoost** learns which specific feature patterns precede labeled incidents. `scale_pos_weight` handles class imbalance; AUCPR is used for early stopping (more informative than AUC-ROC under imbalance).
Weakness: cannot generalize to failure modes unseen during training.

**Ensemble** `score = 0.3 × iso + 0.7 × xgb` — IF provides a safety net for novel failures, XGB keeps precision high.

In [ ]:
train_features, test_features = split_temporal(features, test_ratio=0.2)
train_targets = targets.loc[train_features.index]
test_targets  = targets.loc[test_features.index]

print(f'Train: {train_features.index.min().date()} → {train_features.index.max().date()}  ({len(train_features):,} rows)')
print(f'Test:  {test_features.index.min().date()} → {test_features.index.max().date()}  ({len(test_features):,} rows)')

In [ ]:
from src.models.ensemble import EnsembleModel
from src.models.isolation_forest import IsolationForestModel
from src.models.xgboost_model import XGBoostModel

iso   = IsolationForestModel(n_estimators=100, random_state=42)
xgb   = XGBoostModel(n_estimators=300, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
model = EnsembleModel(
    iso_weight=0.3, xgb_weight=0.7,
    iso_params={'n_estimators': 100, 'random_state': 42},
    xgb_params={'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
                'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42},
)

iso.fit(train_features, train_targets)
xgb.fit(train_features, train_targets)
model.fit(train_features, train_targets)
print('Training complete.')

## 4. Evaluation

### Why incident-level recall instead of point-wise F1?

Point-wise metrics are misleading here:
- A model that always predicts 0 achieves >98% accuracy
- Point-wise recall treats every timestep during an incident as a separate miss — the real question is whether the system warned *before* the incident started

**Incident-level recall**: an incident is *detected* if at least one alert fires before its start. This directly measures advance warning, which is the business objective.

**Lead time**: how many minutes before the incident start the first alert fires.

**False positive rate**: fraction of non-incident timesteps where the score exceeds the threshold — the alert fatigue metric.

In [ ]:
from src.evaluation.metrics import (
    full_evaluation_report,
    compute_recall_at_threshold,
    compute_false_positive_rate,
    compute_lead_time,
)

iso_scores   = iso.predict_proba(test_features)
xgb_scores   = xgb.predict_proba(test_features)
ens_scores   = model.predict_proba(test_features)

print(f'{"Model":<20}  {"ROC-AUC":>8}  {"Recall@0.65":>12}  {"FPR@0.65":>10}')
print('-' * 56)
for name, scores in [('IsolationForest', iso_scores), ('XGBoost', xgb_scores), ('Ensemble', ens_scores)]:
    auc = roc_auc_score(test_targets, scores)
    rec = compute_recall_at_threshold(test_targets, scores, 0.65)
    fpr = compute_false_positive_rate(test_targets, scores, 0.65)
    print(f'{name:<20}  {auc:>8.4f}  {rec:>12.3f}  {fpr:>10.4f}')

IsolationForest alone has higher recall but much higher FPR — routine traffic spikes trigger false alarms. XGBoost alone misses novel failure patterns. The ensemble achieves a better tradeoff.

In [ ]:
report  = full_evaluation_report(test_targets, ens_scores, test_features.index)
opt_t   = report['optimal_threshold_for_recall_0.8']
opt     = report['at_optimal_threshold']
lt      = opt['lead_time']

print(f'ROC-AUC: {report["roc_auc"]:.4f}')
print(f'\nAt threshold {opt_t:.3f} (tuned for 80% recall):')
print(f'  Recall (incident-level): {opt["recall"]:.3f}')
print(f'  False positive rate:     {opt["fpr"]:.4f}')
if lt['mean_lead_time']:
    print(f'  Mean lead time:          {lt["mean_lead_time"]:.1f} min')
    print(f'  Median lead time:        {lt["median_lead_time"]:.1f} min')
    print(f'  Incidents with advance warning: {lt["n_with_lead_time"]} / {lt["n_incidents"]}')

In [ ]:
pr_prec, pr_rec, _ = precision_recall_curve(test_targets, ens_scores)
fpr_roc, tpr_roc, _ = roc_curve(test_targets, ens_scores)

tm = report['threshold_metrics']
t_vals = sorted(tm.keys(), key=float)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(pr_rec, pr_prec, linewidth=2, color='steelblue')
axes[0].axvline(0.8, color='gray', linestyle='--', alpha=0.5, label='Recall = 0.8')
axes[0].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[0].legend()

axes[1].plot([float(t) for t in t_vals], [tm[t]['recall'] for t in t_vals], 'o-', label='Recall', color='green')
axes[1].plot([float(t) for t in t_vals], [tm[t]['fpr']    for t in t_vals], 's-', label='FPR',    color='red')
axes[1].axhline(0.8, color='green', linestyle='--', alpha=0.4)
axes[1].axvline(opt_t, color='gray', linestyle='--', alpha=0.5, label=f'Optimal t={opt_t:.2f}')
axes[1].set(xlabel='Threshold', title='Recall and FPR vs Threshold')
axes[1].legend(fontsize=9)

axes[2].plot(fpr_roc, tpr_roc, linewidth=2, color='purple')
axes[2].plot([0,1],[0,1],'--', color='gray', alpha=0.5)
axes[2].set(xlabel='FPR', ylabel='TPR', title=f'ROC Curve  (AUC={report["roc_auc"]:.3f})')

plt.suptitle('Ensemble Model — Evaluation', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
n = 2880
idx = test_features.index[:n]
sc  = ens_scores[:n]
tgt = test_targets.iloc[:n]
true_inc = df['incident'].loc[idx].values

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

ax1.plot(idx, sc, linewidth=0.8, color='steelblue', label='Ensemble score')
ax1.axhline(opt_t, color='red', linestyle='--', linewidth=1.5, label=f'Threshold ({opt_t:.2f})')
ax1.fill_between(idx, 0, 1, where=tgt.values, alpha=0.2, color='red', label='Incident window (H=15m)')
ax1.set(ylabel='Score', ylim=(0,1), title='Ensemble score vs incident labels — first 2 days of test set')
ax1.legend(loc='upper right', fontsize=9)

ax2.fill_between(idx, 0, 1, where=true_inc,     color='red',    alpha=0.5, label='True incident')
ax2.fill_between(idx, 0, 1, where=sc >= opt_t,  color='orange', alpha=0.35, label='Alert fired')
ax2.set(ylabel='Active', xlabel='Time', title='Alerts vs True Incidents')
ax2.legend(loc='upper right', fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %Hh'))
plt.setp(ax2.get_xticklabels(), rotation=20, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

Orange windows appearing *before* red windows are true advance warnings. The model generally shows elevated scores 5–15 minutes before an incident begins.

In [ ]:
if lt['distribution']:
    plt.figure(figsize=(10, 4))
    plt.hist(lt['distribution'], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    plt.axvline(lt['mean_lead_time'],   color='red',    linestyle='--', label=f'Mean: {lt["mean_lead_time"]:.1f} min')
    plt.axvline(lt['median_lead_time'], color='orange', linestyle='--', label=f'Median: {lt["median_lead_time"]:.1f} min')
    plt.xlabel('Lead Time (minutes)')
    plt.ylabel('Number of incidents')
    plt.title('Alert lead time distribution')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Feature Importance

In [ ]:
importances = pd.Series(
    model.xgb_model._model.feature_importances_,
    index=train_features.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 8))
importances.head(25).sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set(title='Top 25 XGBoost feature importances (gain)', xlabel='Importance')
plt.tight_layout()
plt.show()

print('Top 5 features:')
for feat, val in importances.head(5).items():
    print(f'  {feat:<45s}  {val:.4f}')

Rate-of-change features (e.g., `_roc_1m`) and percentile features (e.g., `_q95_`) dominating the top list confirms that velocity and tail behavior are more predictive than current absolute values.

## 6. Failure Case Analysis

In [ ]:
from src.evaluation.metrics import _get_incident_intervals

intervals = _get_incident_intervals(test_targets)
missed, detected = [], []

for start, end in intervals:
    pre_alert = (ens_scores[:start] >= opt_t).any()
    entry = {
        'duration_min': end - start + 1,
        'max_pre_score': ens_scores[:start].max() if start > 0 else 0.0,
        'peak_score_during': ens_scores[start:end+1].max(),
    }
    (detected if pre_alert else missed).append(entry)

print(f'Detected: {len(detected)}  |  Missed: {len(missed)}  |  Recall: {len(detected)/len(intervals):.3f}')

if missed:
    missed_df = pd.DataFrame(missed)
    print(f'\nMissed incidents:')
    print(f'  Mean duration:      {missed_df["duration_min"].mean():.1f} min')
    print(f'  Mean peak score:    {missed_df["peak_score_during"].mean():.3f}  (threshold={opt_t:.3f})')
    print(f'  Mean max pre-score: {missed_df["max_pre_score"].mean():.3f}')

**Failure patterns:**

1. **Short abrupt incidents** — no ramp-up means no precursor signal; the score spikes *during* the incident, not before
2. **Low-severity incidents** — metric deviations small relative to background noise are indistinguishable from normal variation
3. **Incidents after a recent false positive** — if the alert cooldown is active when a real incident starts, it will be suppressed

These are fundamental limits of the approach, not implementation bugs. Short abrupt failures are better handled by reactive monitoring in parallel with this predictive layer.

### Adaptation to a real system

1. **Labels from existing alarms** — periods when CloudWatch alarms enter ALARM state replace the synthetic `incident` column
2. **Periodic retraining** — a daily job retrains on the most recent 30 days, handling distribution shift from deployments and traffic growth
3. **Threshold recalibration** — the optimal threshold should be re-estimated on held-out production data, not synthetic data
4. **Concept drift monitoring** — compare the current score distribution to the training distribution (e.g., KS test); a large shift triggers immediate retraining
5. **SHAP explanations** — add SHAP values to alert messages so on-call engineers know which metric drove the score